# Notebook 3 — Flowers-102 External Replication

This notebook is the **external-replication / second-dataset notebook** for the RAC paper.

Its purpose is narrow and reviewer-facing:

> Does the separation between residual magnitude and residual correction difficulty also appear outside CIFAR-10?

The notebook deliberately avoids new diffusion sampling. It reuses the already-generated Flowers tensors from the existing project and trains only the small correction networks needed for the replication.

## Primary external-replication test

We focus on the pair:

- DDIM-100 baseline
- DDIM-200 baseline

and correction widths:

- width 4 — moderate capacity
- width 16 — higher capacity

with **5 seeds**.

This directly tests whether the previously observed inversion is:
1. reproducible across random seeds; and
2. stable or weakened at higher correction capacity.

No numerical conclusion is hard-coded. The notebook derives the conclusion from the actual outputs.

In [ ]:
# ============================================================
# GLOBAL IMPORTS + CONFIG
# ============================================================

from pathlib import Path
import random
import copy
import json
import math

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

import matplotlib.pyplot as plt


# ============================================================
# REPRODUCIBILITY
# ============================================================

SEED = 20260927

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# DEVICE
# ============================================================

device = "cuda" if torch.cuda.is_available() else "cpu"


# ============================================================
# PROJECT PATHS
# ============================================================

PROJECT_DIR = Path(
    r"D:\PhD EMP\Paper Writting\STEP_COMPLEXITY"
)

OUT_DIR = PROJECT_DIR / "flowers_external_replication"

# Check project path BEFORE creating anything
if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        f"PROJECT_DIR does not exist:\n{PROJECT_DIR}"
    )

OUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ============================================================
# FLOWERS EXPERIMENT CONFIG
# ============================================================

N_TOTAL = 256

N_TRAIN = 192
N_VAL = 32
N_TEST = 32

TRAIN_END = N_TRAIN
VAL_END = N_TRAIN + N_VAL

BASELINES = [100, 200]

WIDTHS = [4, 16]

SEEDS = [0, 1, 2]


# ============================================================
# TRAINING CONFIG
# ============================================================

BATCH_SIZE = 32
LEARNING_RATE = 1e-3

MAX_EPOCHS = 1000
PATIENCE = 40
MIN_DELTA = 1e-5


# ============================================================
# CHECK EVERYTHING
# ============================================================

print("=" * 60)
print("NOTEBOOK 3 CONFIG")
print("=" * 60)

print("device:", device)

print("\nPROJECT_DIR:")
print(PROJECT_DIR)

print("\nPROJECT_DIR exists:")
print(PROJECT_DIR.exists())

print("\nOUT_DIR:")
print(OUT_DIR)

print("\nOUT_DIR exists:")
print(OUT_DIR.exists())

print("\nSplit:")
print(
    "train =", N_TRAIN,
    "| val =", N_VAL,
    "| test =", N_TEST
)

print("\nBaselines:", BASELINES)
print("Widths:", WIDTHS)
print("Seeds:", SEEDS)

## 1. Locate the saved Flowers tensors automatically

The original Flowers run may have used a slightly different folder name, so this cell searches the project recursively.

We need exactly:
- latent inputs `z`
- DDIM-500 teacher/reference
- DDIM-100 baseline
- DDIM-200 baseline

The code does **not** choose silently if multiple ambiguous files are found. It prints candidates so you can override them explicitly.

In [ ]:
# Search likely Flowers-related tensor files
all_pt = list(PROJECT_DIR.rglob("*.pt"))

flower_candidates = [
    p for p in all_pt
    if any(k in str(p).lower() for k in [
        "flower",
        "flowers",
        "external_replication",
        "flowers64",
    ])
]

print(f"Found {len(flower_candidates)} Flowers-related .pt files:")
for p in sorted(flower_candidates):
    print(" ", p)

In [ ]:
# ------------------------------------------------------------
# Robust candidate selection helpers
# ------------------------------------------------------------

def choose_candidate(candidates, include_terms, exclude_terms=()):
    matches = []

    for p in candidates:
        s = str(p).lower()

        if all(term.lower() in s for term in include_terms):
            if not any(term.lower() in s for term in exclude_terms):
                matches.append(p)

    if len(matches) == 1:
        return matches[0]

    if len(matches) == 0:
        return None

    print("Ambiguous candidates for", include_terms)
    for p in matches:
        print(" ", p)

    return None


# Try common patterns
z_path = (
    choose_candidate(flower_candidates, ["z"])
    or choose_candidate(flower_candidates, ["latent"])
)

teacher_path = (
    choose_candidate(flower_candidates, ["teacher", "500"])
    or choose_candidate(flower_candidates, ["reference", "500"])
)

baseline100_path = (
    choose_candidate(flower_candidates, ["baseline", "100"])
    or choose_candidate(flower_candidates, ["ddim", "100"])
)

baseline200_path = (
    choose_candidate(flower_candidates, ["baseline", "200"])
    or choose_candidate(flower_candidates, ["ddim", "200"])
)

print("\nAUTO-DETECTED PATHS")
print("z:", z_path)
print("teacher500:", teacher_path)
print("baseline100:", baseline100_path)
print("baseline200:", baseline200_path)

### If auto-detection fails

If any path above is `None`, edit only the next cell and insert the exact saved tensor paths from your project.

If auto-detection succeeded for all four, leave the cell unchanged.

In [ ]:
# ============================================================
# OPTIONAL MANUAL OVERRIDES
# Uncomment only if auto-detection above returned None.
# ============================================================

# z_path = Path(r"YOUR_PATH_HERE")
# teacher_path = Path(r"YOUR_PATH_HERE")
# baseline100_path = Path(r"YOUR_PATH_HERE")
# baseline200_path = Path(r"YOUR_PATH_HERE")

required_paths = {
    "z": z_path,
    "teacher500": teacher_path,
    "baseline100": baseline100_path,
    "baseline200": baseline200_path,
}

missing = [k for k, v in required_paths.items() if v is None]

if missing:
    raise FileNotFoundError(
        "Could not auto-detect: "
        + ", ".join(missing)
        + "\nPlease set the corresponding path(s) in the manual override cell."
    )

for name, path in required_paths.items():
    if not Path(path).exists():
        raise FileNotFoundError(f"{name}: file does not exist: {path}")

print("All required Flowers tensors located.")

## 2. Load tensors and validate shapes

In [ ]:
def extract_tensor(obj, file_name):
    if torch.is_tensor(obj):
        return obj

    if isinstance(obj, dict):
        tensor_keys = [
            k for k, v in obj.items()
            if torch.is_tensor(v) and v.ndim == 4
        ]

        if len(tensor_keys) == 1:
            return obj[tensor_keys[0]]

        raise ValueError(
            f"{file_name} contains multiple/zero 4-D tensor candidates: {tensor_keys}"
        )

    raise TypeError(f"Unsupported object in {file_name}: {type(obj)}")


z = extract_tensor(
    torch.load(z_path, map_location="cpu"),
    z_path.name,
).float()

teacher = extract_tensor(
    torch.load(teacher_path, map_location="cpu"),
    teacher_path.name,
).float()

baseline100 = extract_tensor(
    torch.load(baseline100_path, map_location="cpu"),
    baseline100_path.name,
).float()

baseline200 = extract_tensor(
    torch.load(baseline200_path, map_location="cpu"),
    baseline200_path.name,
).float()


for name, x in [
    ("z", z),
    ("teacher", teacher),
    ("baseline100", baseline100),
    ("baseline200", baseline200),
]:
    print(name, x.shape, x.dtype)

    if len(x) != N_TOTAL:
        raise ValueError(
            f"{name}: expected {N_TOTAL} samples, got {len(x)}"
        )

if teacher.shape != baseline100.shape or teacher.shape != baseline200.shape:
    raise ValueError("Teacher and baseline tensor shapes do not match.")

print("\nTensor validation passed.")

## 3. Fixed train / validation / test split

In [ ]:
z_train = z[:TRAIN_END]
z_val = z[TRAIN_END:VAL_END]
z_test = z[VAL_END:]

teacher_train = teacher[:TRAIN_END]
teacher_val = teacher[TRAIN_END:VAL_END]
teacher_test = teacher[VAL_END:]

baselines = {
    100: baseline100,
    200: baseline200,
}

residuals = {
    C: teacher - B
    for C, B in baselines.items()
}

print(
    "train / val / test:",
    len(z_train),
    len(z_val),
    len(z_test),
)

## 4. Baseline residual magnitude

The first half of the external-replication claim is that the 200-step residual is **smaller in absolute magnitude** than the 100-step residual.

This is computed directly from the saved tensors.

In [ ]:
magnitude_rows = []

for C in BASELINES:
    R = residuals[C]

    per_sample_test_mse = (
        (R[VAL_END:] ** 2)
        .flatten(1)
        .mean(dim=1)
    )

    magnitude_rows.append({
        "baseline_steps": C,
        "test_residual_MSE": per_sample_test_mse.mean().item(),
        "test_residual_MSE_SE": (
            per_sample_test_mse.std(unbiased=True).item()
            / math.sqrt(len(per_sample_test_mse))
        ),
        "test_residual_RMS": (
            per_sample_test_mse.mean().sqrt().item()
        ),
    })

df_magnitude = pd.DataFrame(magnitude_rows)

display(df_magnitude)

df_magnitude.to_csv(
    OUT_DIR / "flowers_residual_magnitude.csv",
    index=False,
)

## 5. Correction network

Same local CNN family as the main CIFAR experiment.

In [ ]:
class CorrectionCNN(nn.Module):
    def __init__(self, width):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(3, width, 3, padding=1),
            nn.GELU(),

            nn.Conv2d(width, width, 3, padding=1),
            nn.GELU(),

            nn.Conv2d(width, width, 3, padding=1),
            nn.GELU(),

            nn.Conv2d(width, 3, 3, padding=1),
        )

    def forward(self, x):
        return self.net(x)


def count_params(model):
    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def train_corrector(
    residual,
    width,
    seed,
    max_epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LEARNING_RATE,
    patience=PATIENCE,
    min_delta=MIN_DELTA,
):
    set_seed(seed)

    r_train = residual[:TRAIN_END]
    r_val = residual[TRAIN_END:VAL_END]
    r_test = residual[VAL_END:]

    # Normalize using TRAIN residual only.
    train_rms = torch.sqrt(
        (r_train ** 2).mean()
    ).item()

    y_train = r_train / train_rms

    generator = torch.Generator()
    generator.manual_seed(seed)

    loader = DataLoader(
        TensorDataset(z_train, y_train),
        batch_size=batch_size,
        shuffle=True,
        generator=generator,
    )

    model = CorrectionCNN(width).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
    )

    loss_fn = nn.MSELoss()

    best_val_nmse = float("inf")
    best_state = None
    best_epoch = None
    bad_epochs = 0

    for epoch in range(1, max_epochs + 1):
        model.train()

        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad(set_to_none=True)

            pred = model(xb)
            loss = loss_fn(pred, yb)

            loss.backward()
            optimizer.step()

        model.eval()

        with torch.no_grad():
            pred_val = (
                model(z_val.to(device)).cpu()
                * train_rms
            )

        val_mse = (
            (pred_val - r_val) ** 2
        ).mean().item()

        val_energy = (
            r_val ** 2
        ).mean().item()

        val_nmse = val_mse / val_energy

        if val_nmse < best_val_nmse - min_delta:
            best_val_nmse = val_nmse
            best_state = copy.deepcopy(
                model.state_dict()
            )
            best_epoch = epoch
            bad_epochs = 0
        else:
            bad_epochs += 1

        if bad_epochs >= patience:
            break

    model.load_state_dict(best_state)
    model.eval()

    with torch.no_grad():
        pred_test = (
            model(z_test.to(device)).cpu()
            * train_rms
        )

    test_mse = (
        (pred_test - r_test) ** 2
    ).mean().item()

    test_energy = (
        r_test ** 2
    ).mean().item()

    test_nmse = test_mse / test_energy

    return {
        "width": width,
        "params": count_params(model),
        "seed": seed,
        "best_epoch": best_epoch,
        "val_nmse": best_val_nmse,
        "test_nmse": test_nmse,
    }

## 6. Five-seed external-replication experiment

This is the core reviewer-facing experiment.

Only 20 small-CNN trainings are required:

\[
2\ \text{baselines}
\times
2\ \text{widths}
\times
5\ \text{seeds}
= 20.
\]

In [ ]:
rows = []

for C in BASELINES:
    residual = residuals[C]

    for width in WIDTHS:
        for seed in SEEDS:

            print(
                f"C={C:3d} | "
                f"width={width:2d} | "
                f"seed={seed}"
            )

            result = train_corrector(
                residual=residual,
                width=width,
                seed=seed,
            )

            rows.append({
                "baseline_steps": C,
                **result,
            })


df_raw = pd.DataFrame(rows)

display(df_raw)

df_raw.to_csv(
    OUT_DIR / "flowers_multiseed_raw.csv",
    index=False,
)

In [ ]:
df_raw = pd.read_csv(OUT_DIR / "flowers_multiseed_raw.csv")

## 7. Aggregate the five seeds

In [ ]:
df_summary = (
    df_raw
    .groupby(
        ["baseline_steps", "width", "params"],
        as_index=False,
    )
    .agg(
        test_NMSE_mean=("test_nmse", "mean"),
        test_NMSE_std=("test_nmse", "std"),
        test_NMSE_min=("test_nmse", "min"),
        test_NMSE_max=("test_nmse", "max"),
        seeds=("seed", "count"),
    )
)

display(df_summary)

df_summary.to_csv(
    OUT_DIR / "flowers_multiseed_summary.csv",
    index=False,
)

## 8. Paired seed-wise inversion analysis

For each width we compare:

\[
\Delta =
\mathrm{NMSE}_{200}
-
\mathrm{NMSE}_{100}.
\]

If \(\Delta>0\), the **more accurate 200-step baseline** leaves a **harder normalized residual** at that seed.

This is exactly the external-replication event we care about.

In [ ]:
delta_rows = []

for width in WIDTHS:

    a = (
        df_raw[
            (df_raw["baseline_steps"] == 100)
            & (df_raw["width"] == width)
        ]
        .sort_values("seed")
        [["seed", "test_nmse"]]
        .rename(columns={"test_nmse": "nmse_100"})
    )

    b = (
        df_raw[
            (df_raw["baseline_steps"] == 200)
            & (df_raw["width"] == width)
        ]
        .sort_values("seed")
        [["seed", "test_nmse"]]
        .rename(columns={"test_nmse": "nmse_200"})
    )

    paired = a.merge(
        b,
        on="seed",
        how="inner",
    )

    paired["delta_200_minus_100"] = (
        paired["nmse_200"]
        - paired["nmse_100"]
    )

    for _, r in paired.iterrows():
        delta_rows.append({
            "width": width,
            "seed": int(r["seed"]),
            "nmse_100": r["nmse_100"],
            "nmse_200": r["nmse_200"],
            "delta_200_minus_100": r["delta_200_minus_100"],
        })


df_deltas = pd.DataFrame(delta_rows)

display(df_deltas)

df_deltas.to_csv(
    OUT_DIR / "flowers_seedwise_deltas.csv",
    index=False,
)

## 9. External-replication summary

In [ ]:
replication_rows = []

for width in WIDTHS:

    sub = df_deltas[
        df_deltas["width"] == width
    ]

    delta = sub["delta_200_minus_100"]

    replication_rows.append({
        "width": width,
        "mean_delta_NMSE_200_minus_100": delta.mean(),
        "std_delta": delta.std(ddof=1),
        "positive_seeds": int((delta > 0).sum()),
        "total_seeds": len(delta),
        "fraction_positive": float((delta > 0).mean()),
        "all_positive": bool((delta > 0).all()),
    })


df_replication = pd.DataFrame(replication_rows)

display(df_replication)

df_replication.to_csv(
    OUT_DIR / "flowers_replication_summary.csv",
    index=False,
)

## 10. Automatic reviewer-safe interpretation

This cell deliberately uses conditional logic so the notebook does not claim an inversion unless the observed outputs support it.

In [ ]:
mse100 = float(
    df_magnitude.loc[
        df_magnitude["baseline_steps"] == 100,
        "test_residual_MSE"
    ].iloc[0]
)

mse200 = float(
    df_magnitude.loc[
        df_magnitude["baseline_steps"] == 200,
        "test_residual_MSE"
    ].iloc[0]
)

print("====================================================")
print("EXTERNAL REPLICATION — DATA-DERIVED INTERPRETATION")
print("====================================================")
print()

print(
    f"Residual magnitude: "
    f"C=100 MSE={mse100:.6f}, "
    f"C=200 MSE={mse200:.6f}"
)

if mse200 < mse100:
    print(
        "✓ The 200-step baseline leaves a smaller "
        "absolute residual than the 100-step baseline."
    )
else:
    print(
        "✗ The required residual-magnitude ordering "
        "was NOT observed."
    )

print()

for _, r in df_replication.iterrows():

    width = int(r["width"])
    mean_delta = r["mean_delta_NMSE_200_minus_100"]
    std_delta = r["std_delta"]
    frac = r["fraction_positive"]

    print(
        f"width={width}: "
        f"delta={mean_delta:+.6f} ± {std_delta:.6f}; "
        f"positive seeds={int(r['positive_seeds'])}/{int(r['total_seeds'])}"
    )

    if r["all_positive"] and mean_delta > 0:
        print(
            "  → Robust inversion across all tested seeds."
        )
    elif mean_delta > 0 and frac >= 0.8:
        print(
            "  → Strong but not seed-universal inversion."
        )
    elif abs(mean_delta) <= std_delta:
        print(
            "  → No clear inversion at this capacity; "
            "difference is small relative to seed variability."
        )
    else:
        print(
            "  → The external inversion is not supported "
            "at this capacity."
        )

    print()

## 11. Paper-ready paired-seed figure

Each line corresponds to the **same random seed**.

An upward line from 100 to 200 means:

> the residual became smaller in absolute magnitude, but harder to approximate relatively.

In [ ]:
from pathlib import Path
import gc

import pandas as pd
import matplotlib
matplotlib.use("Agg")   # lightweight non-interactive backend

import matplotlib.pyplot as plt


# ============================================================
# PATH
# ============================================================

PROJECT_DIR = Path(
    r"D:\PhD EMP\Paper Writting\STEP_COMPLEXITY"
)

OUT_DIR = PROJECT_DIR / "flowers_external_replication"

# ============================================================
# LOAD SAVED RESULTS
# ============================================================

df_deltas = pd.read_csv(
    OUT_DIR / "flowers_seedwise_deltas.csv"
)

print(df_deltas)

WIDTHS = sorted(df_deltas["width"].unique())

# ============================================================
# MAKE FIGURES — PNG ONLY
# ============================================================

for width in WIDTHS:

    sub = (
        df_deltas[df_deltas["width"] == width]
        .sort_values("seed")
        .copy()
    )

    print(f"\nPlotting width={width}, rows={len(sub)}")

    fig, ax = plt.subplots(
        figsize=(5.5, 4.2),
        dpi=120
    )

    # Individual paired seeds
    for _, r in sub.iterrows():

        ax.plot(
            [100, 200],
            [r["nmse_100"], r["nmse_200"]],
            marker="o",
            linewidth=1.2,
            alpha=0.7,
        )

    # Mean
    mean100 = sub["nmse_100"].mean()
    mean200 = sub["nmse_200"].mean()

    ax.plot(
        [100, 200],
        [mean100, mean200],
        marker="s",
        linewidth=2.5,
        label="Mean",
    )

    ax.set_xticks([100, 200])

    ax.set_xlabel("DDIM baseline steps")
    ax.set_ylabel("Held-out residual NMSE")

    ax.set_title(
        f"Flowers-102 external replication - width {int(width)}"
    )

    ax.grid(alpha=0.25)
    ax.legend(frameon=False)

    fig.tight_layout()

    save_path = (
        OUT_DIR /
        f"flowers_width{int(width)}_paired_seeds.png"
    )

    fig.savefig(
        save_path,
        dpi=180,
        bbox_inches="tight",
    )

    # IMPORTANT: release memory
    plt.close(fig)
    del fig, ax
    gc.collect()

    print("Saved:", save_path)

print("\nDONE")

## 12. Paper-ready table

In [ ]:
from pathlib import Path
import pandas as pd
import gc

# ============================================================
# PATHS
# ============================================================

PROJECT_DIR = Path(
    r"D:\PhD EMP\Paper Writting\STEP_COMPLEXITY"
)

OUT_DIR = PROJECT_DIR / "flowers_external_replication"

# ============================================================
# LOAD SAVED RESULTS
# ============================================================

summary_path = OUT_DIR / "flowers_multiseed_summary.csv"

if not summary_path.exists():
    raise FileNotFoundError(
        f"Missing file:\n{summary_path}"
    )

df_summary = pd.read_csv(summary_path)

print("Loaded summary:")
print(df_summary.to_string(index=False))


# ============================================================
# FLAT PAPER TABLE
# ============================================================

df100 = (
    df_summary[df_summary["baseline_steps"] == 100]
    [
        [
            "width",
            "params",
            "test_NMSE_mean",
            "test_NMSE_std",
        ]
    ]
    .rename(
        columns={
            "test_NMSE_mean": "NMSE_100_mean",
            "test_NMSE_std": "NMSE_100_std",
        }
    )
)

df200 = (
    df_summary[df_summary["baseline_steps"] == 200]
    [
        [
            "width",
            "params",
            "test_NMSE_mean",
            "test_NMSE_std",
        ]
    ]
    .rename(
        columns={
            "test_NMSE_mean": "NMSE_200_mean",
            "test_NMSE_std": "NMSE_200_std",
        }
    )
)

paper_table = (
    df100
    .merge(
        df200,
        on=["width", "params"],
        how="inner",
    )
    .sort_values("width")
    .reset_index(drop=True)
)

# Add useful delta
paper_table["Delta_200_minus_100"] = (
    paper_table["NMSE_200_mean"]
    - paper_table["NMSE_100_mean"]
)

print("\nPAPER TABLE:")
print(paper_table.to_string(index=False))


# ============================================================
# SAVE CSV
# ============================================================

csv_path = OUT_DIR / "flowers_paper_table.csv"

paper_table.to_csv(
    csv_path,
    index=False
)

print("\nSaved CSV:")
print(csv_path)


# ============================================================
# CREATE SIMPLE LATEX MANUALLY
# ============================================================

lines = []

lines.append(r"\begin{table}[t]")
lines.append(r"\centering")
lines.append(r"\caption{Flowers-102 external replication across correction capacities.}")
lines.append(r"\label{tab:flowers_replication}")
lines.append(r"\begin{tabular}{rrrrr}")
lines.append(r"\toprule")
lines.append(
    r"Width & Params & NMSE$_{100}$ & NMSE$_{200}$ & $\Delta$ \\"
)
lines.append(r"\midrule")

for _, r in paper_table.iterrows():

    nmse100 = (
        f"{r['NMSE_100_mean']:.4f}"
        r" $\pm$ "
        f"{r['NMSE_100_std']:.4f}"
    )

    nmse200 = (
        f"{r['NMSE_200_mean']:.4f}"
        r" $\pm$ "
        f"{r['NMSE_200_std']:.4f}"
    )

    delta = r["Delta_200_minus_100"]

    lines.append(
        f"{int(r['width'])} & "
        f"{int(r['params']):,} & "
        f"{nmse100} & "
        f"{nmse200} & "
        f"{delta:+.4f} \\\\"
    )

lines.append(r"\bottomrule")
lines.append(r"\end{tabular}")
lines.append(r"\end{table}")

latex = "\n".join(lines)

tex_path = OUT_DIR / "flowers_paper_table.tex"

with open(
    tex_path,
    "w",
    encoding="utf-8"
) as f:
    f.write(latex)

print("\nLATEX:")
print(latex)

print("\nSaved LaTeX:")
print(tex_path)

gc.collect()

## 13. Reproducibility manifest

In [ ]:
manifest = {
    "experiment": "Flowers-102 external RAC replication",
    "seed_base": SEED,
    "train_size": N_TRAIN,
    "val_size": N_VAL,
    "test_size": N_TEST,
    "reference": "DDIM-500 Flowers teacher",
    "baselines": BASELINES,
    "widths": WIDTHS,
    "seeds": SEEDS,
    "optimizer": "Adam",
    "learning_rate": LEARNING_RATE,
    "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS,
    "patience": PATIENCE,
    "min_delta": MIN_DELTA,
    "normalization": "train residual RMS only",
    "device": device,
    "pytorch": torch.__version__,
    "input_files": {
        "z": str(z_path),
        "teacher500": str(teacher_path),
        "baseline100": str(baseline100_path),
        "baseline200": str(baseline200_path),
    },
}

with open(
    OUT_DIR / "flowers_manifest.json",
    "w",
) as f:
    json.dump(
        manifest,
        f,
        indent=2,
    )

print("Saved outputs to:")
print(OUT_DIR.resolve())

# What to send back after Run All

Please send:

- `flowers_residual_magnitude.csv`
- `flowers_multiseed_raw.csv`
- `flowers_multiseed_summary.csv`
- `flowers_seedwise_deltas.csv`
- `flowers_replication_summary.csv`
- `flowers_paper_table.csv`
- `flowers_width4_paired_seeds.png`
- `flowers_width16_paired_seeds.png`
- `flowers_manifest.json`

These files are sufficient to decide exactly what external-replication claim the manuscript can support.